# Кластеризация торговых точек по размеру формата

Второй шаг пайплайна (после `01_get_optics.ipynb`, который отбирает точки формата "оптика" отдельно от аптек).

Здесь аптечные точки (без формата "оптика") кластеризуются дважды, разными способами:
1. **по физической площади торгового зала** (`TradeSquare`) - чисто "формат" точки;
2. **по финансовым показателям за период** (выручка + количество чеков) - чисто "бизнес-размер" точки.

Разница между этими двумя разбивками показывает, где физический формат точки не совпадает с её реальной
"весовой категорией" по деньгам - то есть где размер помещения не объясняет объём продаж.

В конце ноутбука - блок самопроверки (`# тест`): сверка посчитанной здесь выручки с уже готовым сводом
показателей, который использовался как контроль правильности расчёта.

Подключение к БД здесь - функция-заглушка, а часть исходных файлов (категории товаров, актуальный реестр
площадей, контрольный свод показателей) не включена в репозиторий - без них финальные ячейки не выполнятся,
но сама логика расчёта видна.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import sqlalchemy

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

from datetime import datetime, timedelta
import gc

pd.options.display.max_columns = 100

## Настройка параметров

In [ ]:
# Число кластеров - используется для обоих раундов кластеризации (по площади и по выручке/чекам)
N_CLUSTERS = 6

# Тип торговой точки "оптика" - исключается здесь (кластеризуются только аптечные точки,
# формат "оптика" обрабатывается отдельно, см. 01_get_optics.ipynb)
OPTICS_TRADE_POINT_TYPE_ID = '4'

# id товарной группы, которая не учитывается в расчёте (не является товарной выручкой в узком смысле)
EXCLUDED_AP_GROUP = 'DD'

In [ ]:
# Периоды для выгрузки чеков. В исходном рабочем ноутбуке даты были сокращены до 1-2 дней
# для быстрого тестового прогона - оставлено как есть, реальный расчёт использует полные периоды.
lb_calc_period_1 = pd.to_datetime('20220201')  # янв 2022 + фев 2022
rb_calc_period_1 = pd.to_datetime('20220202')

lb_calc_period_2 = pd.to_datetime('20220203')  # июн - дек 2022
rb_calc_period_2 = pd.to_datetime('20220204')

lb_calc_period_3 = pd.to_datetime('20220205')  # янв - апр 2023
rb_calc_period_3 = pd.to_datetime('20220206')

## Функции

In [ ]:
def get_db_engine(name: str):
    """Заглушка вместо реального подключения к корпоративному DWH (SQL Server, ODBC) - см. 01_get_optics.ipynb."""
    raise NotImplementedError("Подключение к реальной БД в портфолио-версии не используется")


def batch_load_sql_dwh(query: str) -> pd.DataFrame:
    """Чанкованная загрузка результата SQL-запроса из DWH."""
    CHUNKSIZE = 200_000
    engine = get_db_engine('dwh')
    conn = engine.connect().execution_options(stream_results=True)
    chunks = []
    for chunk_dataframe in pd.read_sql(query, conn, chunksize=CHUNKSIZE):
        chunks.append(chunk_dataframe)
    conn.close()
    return pd.concat(chunks, ignore_index=True)

## Алгоритм выполнения

### Отбор точек и справочник брендов

Все действующие аптечные точки, кроме формата "оптика" - он анализируется отдельно.

In [ ]:
trade_points = pd.read_sql(f"""
    SELECT t1.STORE_LOCATION_ID,
        SUBSTRING(t1.STORE_LOCATION_NM,1,6) as TradePointCode,
        t1.STORE_LOCATION_LVL_ID3,
        t2.PRODUCT_ATTR_NM,
        t1.STORE_LOCATION_LVL_NM1,
        t1.CLOSE_DTTM,
        t1.OPEN_DTTM,
        tp.TradeSquare,
        t1.STORE_LOCATION_NM
    FROM dwh.IN_STORE_LOCATION_NEW t1
    INNER JOIN dwh.IN_CLUSTER_BASKET t2 ON (t1.STORE_LOCATION_LVL_ID3 = t2.STORE_LOCATION_LVL_ID3)
    INNER JOIN dwh.TradePoints tp ON tp.TradePointId = t1.STORE_LOCATION_ID
    WHERE tp.TradePointTypeId != '{OPTICS_TRADE_POINT_TYPE_ID}'
    """, con=get_db_engine('sas_ia'))

trade_points['CLOSE_DTTM'] = pd.to_datetime(trade_points['CLOSE_DTTM'])

Оставляем только точки, открытые на текущий момент и проработавшие не менее 6 месяцев в исследуемом периоде.
Ниже закомментирован вариант с фильтром по одному конкретному бренду сети - в текущей выгрузке берутся все бренды.

In [ ]:
# вариант с фильтром по одному конкретному бренду сети (не используется в текущей выгрузке):
# trade_points = trade_points\
#     .loc[(trade_points.CLOSE_DTTM > rb_calc_period_3) & (trade_points.PRODUCT_ATTR_NM == brand_be)]

trade_points = trade_points\
    .loc[(trade_points.CLOSE_DTTM > rb_calc_period_3)]

trade_points = trade_points.astype({"CLOSE_DTTM": 'datetime64[ns]', "OPEN_DTTM": 'datetime64[ns]'})
trade_points = trade_points.assign(time_dur_to_now=lambda x: (rb_calc_period_3 - x.OPEN_DTTM) / np.timedelta64(1, 'M'))
trade_points = trade_points.query('time_dur_to_now >= 6')
trade_points

In [ ]:
trade_points_brands = pd.read_sql("""
SELECT t2.BrandName, t1.TradePointCode
FROM dwh.TradePoints t1
INNER JOIN dwh.TradePointBrands t2 ON t1.BrandId = t2.BrandId
""", con=get_db_engine('sas_ia'))

### Товарные группы для кластеризации

Список товарных групп берётся из внешнего справочника (в портфолио не включён - без него ячейка ниже не
выполнится). В справочнике категории, помеченные как "удалить из данных", отфильтровываются - остаются
59 групп, которые реально учитываются при кластеризации.

In [ ]:
ap_cat_filtr = pd.read_excel("Категории для кластеризации.xlsx", "Категории для кластеризации")
ap_cat_filtr = ap_cat_filtr.loc[ap_cat_filtr['Категория для кластеризации'] != 'удалить из данных']
groups_to_keep = ap_cat_filtr['2-Группа']
len(groups_to_keep)

### Загрузка чеков

Выгружаются чеки только выбранных точек за заданные периоды, с рядом бизнес-фильтров:
- исключаются опт и возврат опта (оставляются только определённые типы продаж);
- исключаются внутренние операции и нетоварные позиции;
- исключаются заказы через сторонние маркетплейсы/агрегаторы (не через собственные каналы продаж);
- исключаются уценённые товары (срок годности на момент продажи < 6 месяцев);
- исключается отдельная товарная группа 'UN'.

Нужна цена до вычета скидок (`Sum_Price`).

In [ ]:
df = batch_load_sql_dwh("""
SELECT
    fc.ChequeGUId,
    TradePointId,
    fc.ProductId,
    ap.apCode,
    Date,
    Time,
    Sum_Fact,
    Amount,
    Sum_Purch,
    Sum_Price,
    rowNumber as ChequePos,
    ChequeHeadersId,
    voda.Groups
FROM dbo.Fact_cheque fc
  JOIN dbo.InternetOrders io ON fc.InternetOrderId = io.InternetOrderId
  JOIN dbo.InternetOrderTypes iot ON fc.InternetOrderTypeId = iot.InternetOrderTypeId
  JOIN dbo.Parts p ON fc.PartId = p.PartId
  JOIN dbo.Products prod ON prod.ProductId = fc.ProductId
  JOIN dwh.dimAP ap ON ap.apID = prod.apId
  JOIN dbo.v_OLAP_dimAP voda ON voda.apCode = ap.apCode
WHERE fc.IsInternalId = 0
  AND (fc.SaleTypeId = 1 OR fc.SaleTypeId = 3 OR fc.SaleTypeId = 4 OR fc.SaleTypeId = 5)
  AND io.InternetOrderSourceId NOT IN (6, 38)
  AND datediff(month, fc.[Date], p.ExpireDate) >= 6
  AND voda.Class != 'Нетоварные позиции'
  AND TradePointId IN {}
  AND ((Date BETWEEN {} AND {}) OR (DATE BETWEEN {} AND {}) OR (DATE BETWEEN {} AND {}))
  AND voda.Groups != 'UN'
""".format(tuple(trade_points.STORE_LOCATION_ID),
          "'" + str(lb_calc_period_1.date()) + "'", "'" + str(rb_calc_period_1.date()) + "'",
          "'" + str(lb_calc_period_2.date()) + "'", "'" + str(rb_calc_period_2.date()) + "'",
          "'" + str(lb_calc_period_3.date()) + "'", "'" + str(rb_calc_period_3.date()) + "'"))

# оставляем только чеки по товарным группам, прошедшим отбор выше
df = df.query('Groups in @groups_to_keep')

Дополнительно исключается одна товарная группа, которая не считается товарной выручкой в узком смысле.

In [ ]:
dd = batch_load_sql_dwh(f"""SELECT apCode FROM dwh.dimAP da
JOIN dwh.dimAPGroups da2 ON da.apGroupID = da2.apGroupID
WHERE apGroupCode IN ('{EXCLUDED_AP_GROUP}')
""")
df = df[~df.apCode.isin(list(dd.apCode))]

### Агрегаты по точкам: выручка, количество чеков, площадь

In [ ]:
df_size = df.groupby('TradePointId', as_index=False)\
    .agg({'Sum_Price': 'sum', 'ChequeHeadersId': 'count'})
df_size.rename({'Sum_Price': 'Revenue', 'ChequeHeadersId': 'ChequeCount'}, axis=1, inplace=True)
df_size = df_size\
    .merge(trade_points[['STORE_LOCATION_ID', 'TradeSquare']], left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left')
df_size.drop(['STORE_LOCATION_ID'], axis=1, inplace=True)
df_size

Площадь по части точек уточняется по более свежему реестру торговых точек (в портфолио не включён -
содержит реальные адреса/метраж сети). Там, где для точки нашлось уточнённое значение площади - используется оно,
иначе остаётся значение из основной выгрузки.

In [ ]:
sq_new = pd.read_excel('Реестр_ТТ_актуальный.xlsx')
sq_new = sq_new[['Код', 'Актуальная площадь торгового зала']]
sq_new = sq_new.rename({'Код': 'TradePointCode', 'Актуальная площадь торгового зала': 'TradeSquare_new'}, axis=1)

sq_new = sq_new.merge(trade_points[['STORE_LOCATION_ID', 'TradePointCode']], on='TradePointCode', how='left')
sq_new = sq_new.dropna()
# отсекаем служебные строки-заглушки реестра
sq_new = sq_new[sq_new['TradeSquare_new'] != 'нет данных, закрыта/не открыта']

df_size = df_size.merge(sq_new, left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left')
df_size = df_size.drop(['STORE_LOCATION_ID', 'TradePointCode'], axis=1)

df_size['TradeSquare'][~df_size['TradeSquare_new'].isna()] = df_size['TradeSquare_new']
df_size = df_size.drop(['TradeSquare_new'], axis=1)
df_size = df_size.astype({'TradeSquare': 'float64'})
df_size

### Кластеризация по площади (6 кластеров)

Иерархическая кластеризация (Ward) по одному признаку - площади торгового зала. Дендрограмма и "elbow"-график
(вторая производная расстояний) использовались, чтобы визуально проверить, что 6 кластеров - разумный выбор,
после чего кластеры фиксируются через `AgglomerativeClustering`.

In [ ]:
standard = StandardScaler(with_mean=True, with_std=True)

fig = plt.figure(figsize=(15, 9))
shc.dendrogram(shc.linkage(standard.fit_transform(df_size[['TradeSquare']]), method='ward'),
               truncate_mode='lastp', p=N_CLUSTERS)

In [ ]:
fig = plt.figure(figsize=(15, 9))
Z = shc.linkage(standard.fit_transform(df_size[['TradeSquare']]), method='ward')
last = Z[-68:, 2]
last_rev = last[::-1]
idxs = np.arange(1, len(last) + 1)
plt.plot(idxs, last_rev)

acceleration = np.diff(last, 2)  # вторая производная расстояний
acceleration_rev = acceleration[::-1]
plt.plot(idxs[:-2] + 1, acceleration_rev)

plt.axvline(x=N_CLUSTERS, ymax=0.6, color='r', ls=':', lw=2)
plt.show()

In [ ]:
cluster = AgglomerativeClustering(n_clusters=N_CLUSTERS, affinity='euclidean', linkage='ward')
clst_size = cluster.fit_predict(standard.fit_transform(df_size[['TradeSquare']]))

pd.Series(clst_size, name='ClusterId').value_counts()

t-SNE пробовался как альтернативная визуализация распределения кластеров, но в итоговый анализ не пошёл -
оставлено закомментированным как рабочий след.

In [ ]:
# tmp_tsne = TSNE(random_state=2023).fit_transform(standard.fit_transform(df_size[['TradeSquare']]))
# tmp_tsne = pd.DataFrame(tmp_tsne).rename({0: 'x', 1: 'y'}, axis=1)
# tmp_tsne = tmp_tsne.assign(ID_size=pd.Series(clst_size, name='ClusterId'))
# fig = plt.figure(figsize=(15, 9))
# ax = sns.scatterplot(x='x', y='y', data=tmp_tsne, hue='ID_size', legend='full')

### Сборка итоговой таблицы и визуализация по кластерам площади

In [ ]:
result = df_size.reset_index()[['TradePointId', 'Revenue', 'ChequeCount', 'TradeSquare']].copy()
result = result.assign(ID_size_square=pd.Series(clst_size, name='ClusterId'))

result = result.merge(trade_points[['STORE_LOCATION_ID', 'TradePointCode',
                                     'PRODUCT_ATTR_NM', 'STORE_LOCATION_LVL_NM1', 'STORE_LOCATION_NM']],
                       left_on='TradePointId', right_on='STORE_LOCATION_ID', how='left')\
    .drop(['STORE_LOCATION_ID'], axis=1)

result = result.merge(trade_points_brands, how='left', on='TradePointCode')
result = result[['STORE_LOCATION_LVL_NM1', 'STORE_LOCATION_NM', 'TradePointCode',
                  'BrandName', 'Revenue', 'ChequeCount', 'TradeSquare', 'ID_size_square']]
result = result.drop_duplicates('TradePointCode', keep='first')
result

In [ ]:
PALETTE = ['#0000FF', '#800000', '#00FF00', '#000080', '#FF0000', '#FF8C00',
           '#48D1CC', '#4B0082', '#DC143C', '#808080', '#FF00FF']
sns.palplot(sns.color_palette(PALETTE))

In [ ]:
fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_size_square', y='Revenue', hue='BrandName', order=[1, 0, 2, 4, 5, 3], data=result, size=7,
              palette=PALETTE)
plt.show()

In [ ]:
# средний чек = сумма выручки точки / количество чеков в точке
result = result.assign(mean_purch=lambda x: x.Revenue / x.ChequeCount)

fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_size_square', y='mean_purch', hue='BrandName', order=[0, 5, 3, 2, 4, 1], data=result, size=7,
              palette=PALETTE)
plt.show()

### Кластеризация по выручке и числу чеков (6 кластеров)

Тот же подход (Ward, дендрограмма, elbow, AgglomerativeClustering), но уже по двум финансовым признакам сразу.

In [ ]:
fig = plt.figure(figsize=(15, 9))
shc.dendrogram(shc.linkage(standard.fit_transform(df_size[['Revenue', 'ChequeCount']]), method='ward'),
               truncate_mode='lastp', p=N_CLUSTERS)

In [ ]:
fig = plt.figure(figsize=(15, 9))
Z = shc.linkage(standard.fit_transform(df_size[['Revenue', 'ChequeCount']]), method='ward')
last = Z[-68:, 2]
last_rev = last[::-1]
idxs = np.arange(1, len(last) + 1)
plt.plot(idxs, last_rev)

acceleration = np.diff(last, 2)
acceleration_rev = acceleration[::-1]
plt.plot(idxs[:-2] + 1, acceleration_rev)

plt.axvline(x=N_CLUSTERS, ymax=0.6, color='r', ls=':', lw=2)
plt.show()

In [ ]:
cluster = AgglomerativeClustering(n_clusters=N_CLUSTERS, affinity='euclidean', linkage='ward')
clst_rev_cheque = cluster.fit_predict(standard.fit_transform(df_size[['Revenue', 'ChequeCount']]))

pd.Series(clst_rev_cheque, name='ClusterId').value_counts()

In [ ]:
result = result.assign(ID_size_rev_cheque=clst_rev_cheque)

Сравнение полученных кластеров на лог-лог графике выручка/число чеков - целиком и по отдельным подмножествам кластеров.

In [ ]:
fig = plt.figure(figsize=(8, 5))
grid = sns.scatterplot(data=result, x="Revenue", y="ChequeCount", hue='ID_size_rev_cheque')
grid.set(xscale="log", yscale="log")

In [ ]:
fig = plt.figure(figsize=(8, 5))
grid = sns.scatterplot(data=result.query('ID_size_rev_cheque == 2 | ID_size_rev_cheque == 3 | ID_size_rev_cheque == 5'),
                        x="Revenue", y="ChequeCount", hue='ID_size_rev_cheque')
grid.set(xscale="log", yscale="log")

In [ ]:
fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_size_rev_cheque', y='mean_purch', hue='BrandName', order=[1, 4, 3, 0, 5, 2], data=result, size=7,
              palette=PALETTE)
plt.show()

In [ ]:
fig = plt.figure(figsize=(17, 10))
sns.swarmplot(x='ID_size_rev_cheque', y='Revenue', hue='BrandName', order=[1, 4, 3, 0, 5, 2], data=result, size=7,
              palette=PALETTE)
plt.show()

### Результат

In [ ]:
result

In [ ]:
# экспорт результатов (пути к файлам - оригинальные имена, сам экспорт не выполняется в портфолио-версии):
# result.to_excel("size_clustering.xlsx", index=False)
# result.groupby('ID_size_square').describe().to_excel("size_square_clustering_stats.xlsx")
# result.groupby('ID_size_rev_cheque').describe().to_excel("size_rev_cheque_clustering_stats.xlsx")

### Самопроверка

Сверка посчитанной здесь выручки (`Revenue`) с уже готовым сводом показателей по точкам, использовавшимся
как независимый контроль правильности расчёта. Файл со сводом в портфолио не включён - ячейки ниже
демонстрируют сам приём проверки, а не является воспроизводимым расчётом.

In [ ]:
control = pd.read_excel('Свод_показателей_контроль.xlsx', 'all_metrics')

fig = plt.figure(figsize=(15, 9))
shc.dendrogram(shc.linkage(standard.fit_transform(control[['Revenue', 'ChequeCount']]), method='ward'),
               truncate_mode='lastp', p=N_CLUSTERS)

In [ ]:
compare = pd.concat([
    control[['Revenue', 'ChequeCount']].rename({'Revenue': 'Revenue_control', 'ChequeCount': 'ChequeCount_control'}, axis=1),
    df_size,
], axis=1)

grid = sns.scatterplot(data=compare, x="Revenue_control", y="Revenue")
grid.set(xscale="log", yscale="log")

compare = compare.assign(revenue_diff=lambda x: x.Revenue_control - x.Revenue)
compare.revenue_diff.hist(bins=50)